# Blok 12: Wstęp do AI (Machine Learning)

W tym notatniku znajdziesz rozwiązania zadań do bloku:
- od zadań koncepcyjnych (jak zaklasyfikować problem),
- przez inżynierię cech i metryki,
- aż po pierwszy model w bibliotece `scikit-learn`.

Wersja *clean* (do samodzielnej pracy) jest w pliku `cwiczenia_clean.ipynb`.

Rozwiązania traktuj jako **jedną z możliwych dróg**, a nie jedyne słuszne wyjście.
Komentarze w kodzie pomagają zrozumieć poszczególne kroki.


## Opis datasetu

W tym bloku korzystamy z **syntetycznych** (wymyślonych) mini-zbiorów danych, zdefiniowanych bezpośrednio w Pythonie.

### 1. Dataset: `customers_data` – klienci sklepu online (klasyfikacja)

Każdy wiersz reprezentuje jednego użytkownika wizytującego sklep.

Kolumny (klucze w słownikach):
- `age` (int) – wiek użytkownika,
- `country` (str) – kraj pochodzenia (np. "PL", "DE", "UK"),
- `device` (str) – typ urządzenia (np. "mobile", "desktop"),
- `source` (str) – źródło ruchu (np. "ads", "organic", "email"),
- `purchase` (int, 0/1) – czy użytkownik dokonał zakupu (1) czy nie (0).

Zadanie:
- przewidzieć `purchase` na podstawie cech klienta – klasyfikacja binarna.

### 2. Dataset: `apartments_data` – mieszkania (regresja)

Każdy wiersz reprezentuje mieszkanie na sprzedaż.

Kolumny:
- `rooms` (int) – liczba pokoi,
- `area` (float) – powierzchnia mieszkania w metrach kwadratowych,
- `district` (str) – dzielnica (np. "center", "suburbs"),
- `price` (float) – cena mieszkania w tysiącach (np. 350 oznacza 350 000).

Zadanie:
- przewidzieć `price` na podstawie `rooms`, `area`, `district` – regresja.

### 3. Dataset: `churn_true` i `churn_pred` – wyniki modelu churn (klasyfikacja)

Mały, sztuczny przykład do nauki confusion matrix i metryk klasyfikacji.

Listy:
- `churn_true` – prawdziwe etykiety (czy klient odszedł: 1 – odszedł, 0 – został),
- `churn_pred` – przewidywania modelu (taki sam format 0/1).

Na tych listach liczymy TN, FP, FN, TP i metryki: accuracy, precision, recall, F1.

W kolejnej komórce znajduje się kod, który definiuje te przykładowe dane. Uruchom ją na początku pracy z notebookiem.


In [ ]:
# Przygotowanie przykładowych danych syntetycznych
# Uruchom tę komórkę na początku (Shift+Enter)

import numpy as np
import pandas as pd

# Dataset 1: klienci sklepu online
customers_data = [
    {"age": 22, "country": "PL", "device": "mobile",  "source": "ads",     "purchase": 1},
    {"age": 35, "country": "PL", "device": "desktop", "source": "organic", "purchase": 0},
    {"age": 28, "country": "DE", "device": "mobile",  "source": "email",   "purchase": 1},
    {"age": 42, "country": "UK", "device": "desktop", "source": "ads",     "purchase": 0},
    {"age": 30, "country": "PL", "device": "mobile",  "source": "organic", "purchase": 1},
    {"age": 50, "country": "DE", "device": "desktop", "source": "email",   "purchase": 0},
    {"age": 27, "country": "PL", "device": "mobile",  "source": "ads",     "purchase": 1},
    {"age": 45, "country": "UK", "device": "desktop", "source": "organic", "purchase": 0}
]

customers_df = pd.DataFrame(customers_data)

# Dataset 2: mieszkania
apartments_data = [
    {"rooms": 2, "area": 38.0, "district": "center",  "price": 360.0},
    {"rooms": 3, "area": 54.0, "district": "center",  "price": 520.0},
    {"rooms": 2, "area": 45.0, "district": "suburbs", "price": 310.0},
    {"rooms": 4, "area": 70.0, "district": "suburbs", "price": 480.0},
    {"rooms": 1, "area": 28.0, "district": "center",  "price": 290.0},
    {"rooms": 3, "area": 60.0, "district": "suburbs", "price": 430.0}
]

apartments_df = pd.DataFrame(apartments_data)

# Dataset 3: wyniki modelu churn (prawda vs predykcja)
churn_true = [0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0]
churn_pred = [0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1]

print("customers_df (pierwsze wiersze):")
display(customers_df.head())

print("apartments_df (pierwsze wiersze):")
display(apartments_df.head())

print("churn_true:", churn_true)
print("churn_pred:", churn_pred)


# Część A – Typy problemów ML i podstawowe pojęcia

Zadania 1–4 dotyczą podstaw:
- czy dany problem to klasyfikacja, regresja czy coś innego,
- co jest cechą (`X`), a co etykietą (`y`),
- supervised vs unsupervised,
- workflow projektu ML.


## Zadanie 1 – Klasyfikacja czy regresja? (ROZWIĄZANIE)

Idea:
- Jeśli przewidujemy **kategorię** → klasyfikacja.
- Jeśli przewidujemy **liczbę ciągłą** → regresja.
- Jeśli szukamy grup bez etykiet → klasteryzacja.
- Jeśli prognozujemy w czasie → szeregi czasowe.


In [ ]:
# Zadanie 1 – Klasyfikacja czy regresja? (ROZWIĄZANIE)

problems = [
    {
        "name": "Churn prediction",
        "description": "Przewidywanie, czy klient zrezygnuje z usługi w ciągu najbliższego miesiąca.",
        # przewidujemy tak/nie -> klasyfikacja
        "task_type": "classification"
    },
    {
        "name": "House price prediction",
        "description": "Przewidywanie ceny mieszkania na podstawie liczby pokoi, metrażu i lokalizacji.",
        # cena jest liczbą ciągłą -> regresja
        "task_type": "regression"
    },
    {
        "name": "Customer segmentation",
        "description": "Podział klientów na grupy o podobnym zachowaniu zakupowym bez znanych etykiet.",
        # brak etykiet, szukamy grup -> klasteryzacja
        "task_type": "clustering"
    },
    {
        "name": "Sales forecast",
        "description": "Prognozowanie dziennej sprzedaży sklepu na najbliższe 30 dni.",
        # prognoza w czasie -> szeregi czasowe
        "task_type": "time_series"
    }
]

for p in problems:
    print(p["name"], "->", p["task_type"])


## Zadanie 2 – Cechy (X) vs etykieta (y) (ROZWIĄZANIE)

Chcemy przewidzieć, **czy użytkownik kupi produkt** (`purchase`) na podstawie jego cech (`age`, `country`, `device`, `source`).

- `X` – age, country, device, source.
- `y` – purchase (0/1).


In [ ]:
# Zadanie 2 – Cechy vs etykieta (ROZWIĄZANIE)

# Chcemy przewidzieć, czy klient dokona zakupu (purchase)
# na podstawie wieku, kraju, urządzenia i źródła ruchu.

X_customers = customers_df[["age", "country", "device", "source"]]
y_customers = customers_df["purchase"]

print("X_customers shape:", X_customers.shape)
print("y_customers shape:", y_customers.shape)
display(X_customers.head())
display(y_customers.head())


## Zadanie 3 – Supervised vs unsupervised (ROZWIĄZANIE)

Klucz:
- **supervised** – mamy etykiety (np. churn, fraud),
- **unsupervised** – mamy tylko cechy, chcemy odkryć strukturę.


In [ ]:
# Zadanie 3 – Supervised vs unsupervised (ROZWIĄZANIE)

scenarios = [
    {
        "description": "Mamy dane historyczne: cechy klienta i informację, czy zrezygnował z usługi. Chcemy przewidywać churn.",
        "learning_type": "supervised"  # mamy etykietę: churn
    },
    {
        "description": "Mamy tylko dane o zachowaniu klientów w sklepie (odsłony, kliknięcia, ścieżki). Chcemy znaleźć segmenty klientów.",
        "learning_type": "unsupervised"  # brak etykiet, segmentacja
    },
    {
        "description": "Mamy dane o transakcjach i informację, czy były fraudem. Chcemy nauczyć model wykrywania fraudu.",
        "learning_type": "supervised"  # mamy etykietę: fraud
    }
]

for s in scenarios:
    print("Opis:", s["description"])
    print("Rodzaj uczenia:", s["learning_type"])
    print("-")


## Zadanie 4 – Workflow ML na prostym case (ROZWIĄZANIE)

Case: przewidujemy, czy użytkownik kupi produkt w sklepie online.
Poniżej przykładowa sekwencja kroków workflow ML.


In [ ]:
# Zadanie 4 – Workflow ML (ROZWIĄZANIE)

workflow_steps = [
    "Zdefiniowanie problemu biznesowego: przewidzieć, czy użytkownik kupi produkt podczas wizyty.",
    "Zebranie i integracja danych o użytkownikach (wiek, kraj, urządzenie, źródło ruchu, informacja o zakupie).",
    "Wstępne sprawdzenie jakości danych (braki, duplikaty, zakresy wartości).",
    "Eksploracyjna analiza danych (rozkłady cech, zależności między cechami a zakupem).",
    "Wybór cech wejściowych (np. age, country, device, source) i definicja etykiety (purchase).",
    "Przygotowanie cech: kodowanie kategorii (one-hot encoding), ewentualne skalowanie cech numerycznych.",
    "Podział danych na zbiór treningowy i testowy (np. 80/20).",
    "Wybór modelu bazowego (np. logistic regression) i metryki (np. accuracy, precision, recall, F1).",
    "Wytrenowanie modelu na zbiorze treningowym (fit).",
    "Ocena modelu na zbiorze testowym (predict + liczenie metryk).",
    "Interpretacja wyników: czy metryki są wystarczające z perspektywy biznesu?",
    "Iteracje: poprawa cech, ewentualny wybór innego modelu, regularyzacja, tuning parametrów."
]

for i, step in enumerate(workflow_steps, start=1):
    print(f"Krok {i}:", step)


# Część B – Inżynieria cech

Zadania 5–6 pokazują:
- one-hot encoding,
- skalowanie / standaryzację.


## Zadanie 5 – One-hot encoding na małym zbiorze (ROZWIĄZANIE)

Dla `device` (mobile/desktop) chcemy utworzyć kolumny:
- `device_mobile`,
- `device_desktop`.

Użyjemy `pd.get_dummies`.


In [ ]:
# Zadanie 5 – One-hot encoding (ROZWIĄZANIE)

# One-hot encoding dla device, a dodatkowo dla country i source.

customers_encoded = pd.get_dummies(
    customers_df,
    columns=["device", "country", "source"],
    drop_first=False  # zachowujemy wszystkie kolumny, bo zbiór jest mały
)

print("Oryginalne kolumny:", list(customers_df.columns))
print("Zakodowane kolumny:", list(customers_encoded.columns))
display(customers_encoded.head())


## Zadanie 6 – Skalowanie i standaryzacja cech (ROZWIĄZANIE)

Pracujemy na kolumnie `area` z `apartments_df`.

- liczymy min, max, mean, std,
- stosujemy min-max scaling,
- stosujemy standaryzację.


In [ ]:
# Zadanie 6 – Skalowanie i standaryzacja (ROZWIĄZANIE)

area = apartments_df["area"]

# Obliczenie podstawowych statystyk
area_min = area.min()
area_max = area.max()
area_mean = area.mean()
area_std = area.std(ddof=0)  # wariancja populacyjna (ddof=0) dla uproszczenia

print("area_min:", area_min)
print("area_max:", area_max)
print("area_mean:", area_mean)
print("area_std:", area_std)

# Min-max scaling
area_minmax = (area - area_min) / (area_max - area_min)

# Standaryzacja (z-score)
area_standard = (area - area_mean) / area_std

print("\nOryginalne area:")
print(area.values)

print("\narea po min-max scalingu:")
print(area_minmax.values)

print("\narea po standaryzacji:")
print(area_standard.values)

# Analogicznie można powtórzyć dla rooms (opcjonalne):
rooms = apartments_df["rooms"]
rooms_min = rooms.min()
rooms_max = rooms.max()
rooms_minmax = (rooms - rooms_min) / (rooms_max - rooms_min)
print("\nrooms:", rooms.values)
print("rooms po min-max:", rooms_minmax.values)


# Część C – Metryki klasyfikacji i regresji

Zadania 7–9:
- confusion matrix, accuracy, precision, recall, F1,
- MSE, RMSE, MAE.


## Zadanie 7 – Confusion matrix, accuracy, precision, recall, F1 (ROZWIĄZANIE)

Założyliśmy, że klasa pozytywna to 1 (klient odchodzi).
Poniżej ręczne zliczanie TN, FP, FN, TP i liczenie metryk.


In [ ]:
# Zadanie 7 – Ręczne liczenie metryk klasyfikacji (ROZWIĄZANIE)

print("churn_true:", churn_true)
print("churn_pred:", churn_pred)

TN = 0
FP = 0
FN = 0
TP = 0

# Iterujemy po parach (prawda, predykcja)
for yt, yp in zip(churn_true, churn_pred):
    if yt == 0 and yp == 0:
        TN += 1
    elif yt == 0 and yp == 1:
        FP += 1
    elif yt == 1 and yp == 0:
        FN += 1
    elif yt == 1 and yp == 1:
        TP += 1

print("TN:", TN, "FP:", FP, "FN:", FN, "TP:", TP)

total = TN + FP + FN + TP
accuracy = (TP + TN) / total if total > 0 else 0.0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


## Zadanie 8 – Interpretacja confusion matrix (model fraud) (ROZWIĄZANIE)

Liczymy precision i recall dla modeli A i B, a potem je porównujemy.

- Model A ma mało FP, ale bardzo mało TP (niski recall).
- Model B ma więcej FP, ale dużo wyższy recall.

W kontekście fraudów często ważniejszy jest **wysoki recall** – lepiej mieć więcej fałszywych alarmów niż przeoczyć fraud.


In [ ]:
# Zadanie 8 – Interpretacja confusion matrix (ROZWIĄZANIE)

TN_A, FP_A, FN_A, TP_A = 980, 10, 15, 5
TN_B, FP_B, FN_B, TP_B = 950, 40, 5, 25

precision_A = TP_A / (TP_A + FP_A) if (TP_A + FP_A) > 0 else 0.0
recall_A = TP_A / (TP_A + FN_A) if (TP_A + FN_A) > 0 else 0.0

precision_B = TP_B / (TP_B + FP_B) if (TP_B + FP_B) > 0 else 0.0
recall_B = TP_B / (TP_B + FN_B) if (TP_B + FN_B) > 0 else 0.0

print("Model A – precision:", precision_A, "recall:", recall_A)
print("Model B – precision:", precision_B, "recall:", recall_B)

# Komentarz (tekstowo):
comment = (
    "Model A ma bardzo wysoką precyzję (mało fałszywych alarmów), "
    "ale bardzo niską czułość – przeocza większość fraudów. "
    "Model B ma niższą precyzję (więcej fałszywych alarmów), ale dużo wyższy recall, "
    "czyli wykrywa dużo więcej fraudów. W typowym scenariuszu bezpieczeństwa wolałbym model B, "
    "bo przeoczenie fraudu jest bardziej kosztowne niż dodatkowa weryfikacja kilku poprawnych transakcji."
)
print("\nKomentarz:")
print(comment)


## Zadanie 9 – Metryki regresji: MSE, RMSE, MAE (ROZWIĄZANIE)

Na małym przykładzie liczymy MSE, RMSE i MAE.

- MSE/RMSE mocniej karzą duże błędy (bo błąd jest do kwadratu),
- MAE liczy średnią z modułów błędów.


In [ ]:
# Zadanie 9 – Metryki regresji (ROZWIĄZANIE)

y_true_prices = [360.0, 520.0, 310.0, 480.0, 290.0]
y_pred_prices = [350.0, 510.0, 400.0, 450.0, 280.0]

y_true_arr = np.array(y_true_prices)
y_pred_arr = np.array(y_pred_prices)

errors = y_true_arr - y_pred_arr
sq_errors = errors ** 2

mse = np.mean(sq_errors)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(errors))

print("y_true:", y_true_prices)
print("y_pred:", y_pred_prices)
print("Błędy (y - y_hat):", errors)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)

print("\nKomentarz:")
print("MSE i RMSE silniej karzą duże błędy, bo błąd jest podniesiony do kwadratu.")
print("MAE traktuje wszystkie błędy liniowo, przez co jest bardziej odporne na pojedyncze skrajne wartości (outliery).")


# Część D – Pierwszy model w scikit-learn i workflow

Zadania 10–11:
- pierwszy model w `scikit-learn`,
- opis workflow ML dla mini-case'u.


## Zadanie 10 – Pierwszy model w scikit-learn (ROZWIĄZANIE – wariant klasyfikacji)

Wybieramy wariant **klasyfikacji** na `customers_df`:

1. Przygotowujemy cechy (po one-hot encodingu) i etykietę `purchase`.
2. Dzielimy dane na train/test.
3. Trenujemy `LogisticRegression`.
4. Liczymy accuracy.

Dla własnego treningu możesz spróbować samodzielnie wersji regresyjnej na `apartments_df`.


In [ ]:
# Zadanie 10 – Pierwszy model w scikit-learn (ROZWIĄZANIE – klasyfikacja)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Przygotowanie X i y: one-hot encoding zmiennych kategorycznych
customers_for_model = pd.get_dummies(
    customers_df,
    columns=["country", "device", "source"],
    drop_first=False
)

X = customers_for_model.drop(columns=["purchase"])
y = customers_for_model["purchase"]

print("Kolumny cech:", list(X.columns))
print("X shape:", X.shape)

# 2. Podział na train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# 3. Utworzenie i trening modelu
model = LogisticRegression(solver="liblinear")  # solver dobry dla małych zbiorów
model.fit(X_train, y_train)

# 4. Predykcja i metryka
y_pred = model.predict(X_test)
metric_value = accuracy_score(y_test, y_pred)

print("Predykcje:", y_pred)
print("Rzeczywiste:", y_test.values)
print("Accuracy:", metric_value)

print("\nKomentarz:")
print("Accuracy na tak małym, sztucznym zbiorze nie jest super miarodajne, ale pokazuje pełen przepływ: "
      "przygotowanie cech -> train_test_split -> fit -> predict -> metryka.")


## Zadanie 11 – Workflow ML na mini-case (ROZWIĄZANIE)

Case: klasyfikacja `purchase` w `customers_df`.
Poniżej opis słowny kroków, które wykonaliśmy (lub moglibyśmy wykonać) w takim projekcie.


In [ ]:
# Zadanie 11 – Workflow ML na mini-case (ROZWIĄZANIE)

chosen_case = "classification_customers"

workflow_description = [
    "1. Zdefiniowanie problemu: przewidzieć, czy klient dokona zakupu (purchase = 0/1) podczas wizyty w sklepie.",
    "2. Identyfikacja dostępnych danych: wiek, kraj, typ urządzenia, źródło ruchu, informacja czy kupił.",
    "3. Wstępna analiza danych: sprawdzenie liczebności, rozkładów cech, balans klas (ile 0 vs 1).",
    "4. Wybór cech i etykiety: X = age, country, device, source; y = purchase.",
    "5. Przygotowanie cech: one-hot encoding dla zmiennych kategorycznych, ewentualne skalowanie (tu pomijamy).",
    "6. Podział zbioru na część treningową i testową, np. 75%/25%.",
    "7. Wybór modelu bazowego: logistic regression (prosty, interpretowalny).",
    "8. Wybór metryk: accuracy jako punkt wyjścia, a przy poważnym case także precision/recall/F1.",
    "9. Trening modelu na zbiorze treningowym: fit(X_train, y_train).",
    "10. Predykcja na zbiorze testowym i wyliczenie metryk (accuracy itp.).",
    "11. Interpretacja wyników: czy model jest lepszy niż losowy, czy spełnia wymagania biznesu.",
    "12. Kolejne iteracje: modyfikacja cech, sprawdzenie innych modeli, tuning hiperparametrów, jeśli to potrzebne."
]

for step in workflow_description:
    print(step)


# Część E – Praca domowa (opcjonalna) – Zadania 12 i 13 (ROZWIĄZANIA)

Poniżej przykładowe rozwiązania zadań domowych.
Możesz traktować je jako inspirację lub punkt odniesienia do własnej implementacji.


## Zadanie 12 – Mini-pipeline klasyfikacji (ROZWIĄZANIE)

Budujemy mini-pipeline klasyfikacyjny dla `customers_df`:
- one-hot encoding dla zmiennych kategorycznych,
- podział train/test,
- model `RandomForestClassifier`,
- liczenie accuracy, precision, recall, F1.


In [ ]:
# Zadanie 12 – Mini-pipeline klasyfikacji (ROZWIĄZANIE)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. One-hot encoding zmiennych kategorycznych
customers_for_clf = pd.get_dummies(
    customers_df,
    columns=["country", "device", "source"],
    drop_first=False
)

X_clf = customers_for_clf.drop(columns=["purchase"])
y_clf = customers_for_clf["purchase"]

print("Kolumny X_clf:", list(X_clf.columns))

# 2. Podział na train/test
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=42
)

print("X_train_clf shape:", X_train_clf.shape)
print("X_test_clf shape:", X_test_clf.shape)

# 3. Wybór modelu i trening
clf_model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)
clf_model.fit(X_train_clf, y_train_clf)

# 4. Predykcja i metryki
y_pred_clf = clf_model.predict(X_test_clf)

acc = accuracy_score(y_test_clf, y_pred_clf)
prec = precision_score(y_test_clf, y_pred_clf, zero_division=0)
rec = recall_score(y_test_clf, y_pred_clf, zero_division=0)
f1score = f1_score(y_test_clf, y_pred_clf, zero_division=0)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1score)

print("\nKomentarz:")
print("Na tak małym zbiorze wyniki będą niestabilne, ale w realnym case ważne jest, by dobrać metrykę do problemu. "
      "Np. przy rzadkich zakupach accuracy może być mylące, a ważniejsze będą precision/recall/F1.")


## Zadanie 13 – Bias vs variance na wykresie (ROZWIĄZANIE – przykładowe)

Eksperyment:
- generujemy dane z funkcji sinus z szumem,
- dopasowujemy regresję wielomianową o różnych stopniach,
- liczymy MSE na train i test,
- rysujemy wykres błędu vs stopień wielomianu.

Interpretacja:
- niski stopień: underfitting (wysoki błąd wszędzie),
- bardzo wysoki stopień: overfitting (niski błąd train, wysoki test),
- gdzieś pośrodku – sensowny kompromis bias/variance.


In [ ]:
# Zadanie 13 – Bias vs variance na wykresie (ROZWIĄZANIE)

import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 1. Generujemy dane 1D: y = sin(2*pi*x) + szum
np.random.seed(42)
n_samples = 100
X_all = np.random.rand(n_samples, 1)  # x w [0, 1]
y_all = np.sin(2 * np.pi * X_all[:, 0]) + 0.3 * np.random.randn(n_samples)

X_train_bv, X_test_bv, y_train_bv, y_test_bv = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42
)

# 2. Różne stopnie wielomianu
degrees = [1, 3, 5, 9]
train_errors = []
test_errors = []

for d in degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=d, include_bias=False)),
        ("linreg", LinearRegression())
    ])
    model.fit(X_train_bv, y_train_bv)

    y_train_pred = model.predict(X_train_bv)
    y_test_pred = model.predict(X_test_bv)

    mse_train = mean_squared_error(y_train_bv, y_train_pred)
    mse_test = mean_squared_error(y_test_bv, y_test_pred)

    train_errors.append(mse_train)
    test_errors.append(mse_test)

plt.figure(figsize=(6, 4))
plt.plot(degrees, train_errors, marker="o", label="MSE train")
plt.plot(degrees, test_errors, marker="s", label="MSE test")
plt.xlabel("Stopień wielomianu")
plt.ylabel("MSE")
plt.title("Bias vs variance – przykład regresji wielomianowej")
plt.legend()
plt.grid(True)
plt.show()

print("Stopnie:", degrees)
print("MSE train:", train_errors)
print("MSE test:", test_errors)

print("\nKomentarz:")
print("Przy niskim stopniu (np. 1) model jest zbyt prosty – ma wysoki błąd na train i test (underfitting, wysoki bias).")
print("Przy bardzo wysokim stopniu (np. 9) błąd na train może być bardzo niski, ale błąd na test rośnie (overfitting, wysoka variance).")
print("Pośrodku (np. stopień 3 lub 5) często dostajemy lepszy kompromis między bias i variance.")
